# Python OOP (Object-Oriented Programming)

A hands-on, runnable reference for Python's OOP concepts. Run each code cell as you go — feel free to tweak values and re-run.


## 1. Classes and Objects

- A **class** is a blueprint for creating objects. An **object** (instance) is a concrete thing built from that blueprint.
- Define a class with the `class` keyword. By convention, class names use `PascalCase`.
- Calling a class like a function creates a new instance.


In [ ]:
class Dog:
    """A simple class with no attributes or methods yet."""
    pass

# Creating objects (instances) of the Dog class
d1 = Dog()
d2 = Dog()

print(type(d1))
print(d1 is d2)  # each instance is a distinct object in memory


## 2. The `__init__` Constructor and `self`

- `__init__` is a special method (a *dunder* — double underscore — method) that runs automatically when an object is created.
- `self` refers to the instance being created/used. It's always the first parameter of instance methods, and Python passes it automatically.
- Attributes assigned to `self` inside `__init__` become **instance attributes**.


In [ ]:
class Dog:
    def __init__(self, name, breed):
        self.name = name    # instance attribute
        self.breed = breed  # instance attribute

d1 = Dog("Rex", "Labrador")
d2 = Dog("Fido", "Poodle")

print(d1.name, d1.breed)
print(d2.name, d2.breed)


## 3. Instance Methods

- Regular functions defined inside a class that operate on an instance via `self`.
- They can read/modify instance attributes and call other methods on the same object.


In [ ]:
class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed
        self.tricks = []

    def bark(self):
        return f"{self.name} says Woof!"

    def learn_trick(self, trick):
        self.tricks.append(trick)

d1 = Dog("Rex", "Labrador")
print(d1.bark())

d1.learn_trick("sit")
d1.learn_trick("roll over")
print(d1.tricks)


## 4. Class Attributes vs Instance Attributes

- **Class attributes** are defined directly in the class body and are shared by *all* instances (unless overridden).
- **Instance attributes** are set on `self` and belong to that specific object only.
- Careful: mutating a shared *mutable* class attribute (like a list) through one instance affects all instances — use instance attributes for per-object state.


In [ ]:
class Dog:
    species = "Canis familiaris"  # class attribute, shared by all Dogs

    def __init__(self, name):
        self.name = name  # instance attribute, unique per Dog

d1 = Dog("Rex")
d2 = Dog("Fido")

print(d1.species, d2.species)      # both share the same class attribute
print(Dog.species)                 # accessible via the class itself

d1.species = "Custom species"      # this creates a new INSTANCE attribute on d1,
                                    # it does not change the class attribute
print(d1.species, d2.species, Dog.species)


## 5. Class Methods and Static Methods

- `@classmethod` methods receive the class (`cls`) instead of the instance. Commonly used for alternative constructors.
- `@staticmethod` methods receive neither `self` nor `cls` — they're just regular functions namespaced inside the class, grouped there because they're logically related.


In [ ]:
class Dog:
    species = "Canis familiaris"

    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year, current_year=2026):
        """Alternative constructor: build a Dog from a birth year instead of an age."""
        age = current_year - birth_year
        return cls(name, age)

    @staticmethod
    def is_adult(age):
        """A pure utility function related to Dog, but needs no instance/class state."""
        return age >= 1

d1 = Dog.from_birth_year("Rex", 2023)
print(d1.name, d1.age)
print(Dog.is_adult(d1.age))


## 6. Encapsulation: Public, Protected, and Private Attributes

- Python has no true "private" enforcement — it relies on naming conventions:
  - `name` — public, freely accessible.
  - `_name` — protected *by convention*: "internal use, but not enforced". Other code should treat it as an implementation detail.
  - `__name` — private-ish: Python performs **name mangling** to `_ClassName__name`, making accidental access/collisions in subclasses harder (not truly private).


In [ ]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner            # public
        self._account_type = "savings"  # protected (convention)
        self.__balance = balance        # "private" (name-mangled)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.__balance += amount

    def get_balance(self):
        return self.__balance

acc = BankAccount("Deepak", 1000)
acc.deposit(500)
print(acc.get_balance())

print(acc._account_type)         # accessible, but "please don't touch" by convention

try:
    print(acc.__balance)         # raises AttributeError — not found under this name
except AttributeError as e:
    print("Error:", e)

print(acc._BankAccount__balance)  # name mangling reveals the real attribute name


## 7. Properties (Pythonic Getters and Setters)

- Instead of writing explicit `get_x()`/`set_x()` methods, Python uses the `@property` decorator so attribute-style access (`obj.x`) can run validation logic behind the scenes.
- `@x.setter` defines what happens on assignment; `@x.deleter` defines what happens on `del obj.x`.


In [ ]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance

    @property
    def balance(self):
        """Getter: runs when you read acc.balance"""
        return self.__balance

    @balance.setter
    def balance(self, value):
        """Setter: runs when you write acc.balance = value"""
        if value < 0:
            raise ValueError("Balance cannot be negative")
        self.__balance = value

acc = BankAccount("Deepak", 1000)
print(acc.balance)      # calls the getter, looks like plain attribute access

acc.balance = 2000       # calls the setter
print(acc.balance)

try:
    acc.balance = -50    # setter validation kicks in
except ValueError as e:
    print("Error:", e)


## 8. Inheritance

- Inheritance lets a class (**child/subclass**) reuse and extend the behavior of another class (**parent/superclass**) — an "is-a" relationship.
- `super()` gives access to the parent class's methods, most commonly used inside `__init__` to initialize inherited attributes.


In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound"

    def describe(self):
        return f"I am {self.name}, a {type(self).__name__}"


class Dog(Animal):                # Dog inherits from Animal
    def __init__(self, name, breed):
        super().__init__(name)    # call the parent's __init__
        self.breed = breed

    def speak(self):              # method overriding, see section 9
        return f"{self.name} says Woof!"


a = Animal("Generic Animal")
d = Dog("Rex", "Labrador")

print(a.speak())
print(d.speak())        # overridden behavior
print(d.describe())     # inherited, unchanged behavior
print(isinstance(d, Animal))  # a Dog IS-A Animal


## 9. Method Overriding and Polymorphism

- **Overriding**: a subclass redefines a method it inherited, to change or specialize its behavior (see `Dog.speak` above).
- **Polymorphism**: objects of different classes can be used interchangeably if they share a common interface — the same method call produces type-appropriate behavior.


In [ ]:
class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"


class Cow(Animal):
    def speak(self):
        return f"{self.name} says Moo!"


animals = [Dog("Rex", "Labrador"), Cat("Whiskers"), Cow("Bessie")]

# Polymorphism: same method call (.speak()), different behavior per object type
for animal in animals:
    print(animal.speak())


## 10. Abstraction (Abstract Base Classes)

- Abstraction means exposing only the essential interface and hiding implementation details.
- The `abc` module lets you define **abstract base classes** with `@abstractmethod` — methods that subclasses *must* implement. An abstract class cannot be instantiated directly.


In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        """Subclasses must implement this."""
        ...

    @abstractmethod
    def perimeter(self):
        ...

    def describe(self):
        # a concrete (non-abstract) method — shared by all subclasses for free
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


try:
    s = Shape()               # cannot instantiate an abstract class
except TypeError as e:
    print("Error:", e)

r = Rectangle(4, 5)
print(r.describe())


## 11. Dunder (Magic) Methods

- "Dunder" = double underscore. These special methods let your objects integrate with Python's built-in syntax and functions (`print()`, `len()`, `==`, `+`, `for`, etc.).
- `__str__` — human-readable string, used by `print()`/`str()`.
- `__repr__` — unambiguous developer-facing string, used in the REPL/debugger and as a fallback for `str()`.
- `__eq__`, `__lt__`, ... — comparison operators.
- `__len__`, `__getitem__`, `__iter__` — make objects work with `len()`, indexing, and `for` loops.


In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"           # print(v) -> "(1, 2)"

    def __repr__(self):
        return f"Vector(x={self.x}, y={self.y})"  # repr(v) / typing v in REPL

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

    def __add__(self, other):                     # operator overloading, see section 12
        return Vector(self.x + other.x, self.y + other.y)

    def __len__(self):
        return int((self.x ** 2 + self.y ** 2) ** 0.5)


v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1)          # uses __str__
print(repr(v1))     # uses __repr__
print(v1 == Vector(1, 2))  # uses __eq__
print(v1 + v2)      # uses __add__, result printed via __str__
print(len(v2))      # uses __len__


## 12. Operator Overloading

- Operator overloading is simply implementing the right dunder method so a built-in operator (`+`, `-`, `*`, `==`, `<`, ...) works meaningfully for your class.
- Common mappings: `+` → `__add__`, `-` → `__sub__`, `*` → `__mul__`, `<` → `__lt__`, `in` → `__contains__`.


In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    def __add__(self, other):
        return Money(self.amount + other.amount)

    def __sub__(self, other):
        return Money(self.amount - other.amount)

    def __lt__(self, other):
        return self.amount < other.amount

    def __str__(self):
        return f"${self.amount:.2f}"


wallet = Money(50)
price = Money(20)

print(wallet + price)   # __add__
print(wallet - price)   # __sub__
print(price < wallet)   # __lt__


## 13. Composition vs Inheritance

- **Inheritance** models "is-a" relationships (a `Dog` IS-A `Animal`).
- **Composition** models "has-a" relationships — a class holds an instance of another class as an attribute and delegates to it. Often preferred over inheritance because it's more flexible and avoids deep, fragile class hierarchies ("favor composition over inheritance").


In [ ]:
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower

    def start(self):
        return f"Engine starting with {self.horsepower}HP"


class Car:
    def __init__(self, make, horsepower):
        self.make = make
        self.engine = Engine(horsepower)  # Car HAS-A Engine (composition)

    def start(self):
        return f"{self.make}: {self.engine.start()}"


car = Car("Toyota", 150)
print(car.start())


## 14. Multiple Inheritance and the Method Resolution Order (MRO)

- A Python class can inherit from more than one parent class.
- When multiple parents define the same method, Python decides which one to use via the **MRO** (Method Resolution Order), computed with the C3 linearization algorithm. Inspect it with `ClassName.__mro__` or `ClassName.mro()`.


In [ ]:
class Flyer:
    def move(self):
        return "Flying"


class Swimmer:
    def move(self):
        return "Swimming"


class Duck(Flyer, Swimmer):  # multiple inheritance: order matters for the MRO
    pass


d = Duck()
print(d.move())              # uses Flyer.move because Flyer is listed first
print(Duck.__mro__)          # shows the exact lookup order Python follows


## 15. `__slots__`

- By default, instances store their attributes in a per-object `__dict__`, which is flexible but uses extra memory.
- Declaring `__slots__` restricts an instance to a fixed set of attributes, saving memory and slightly speeding up attribute access — useful when creating many small objects.


In [ ]:
class Point:
    __slots__ = ("x", "y")  # no per-instance __dict__; only x and y are allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(1, 2)
print(p.x, p.y)

try:
    p.z = 3   # not declared in __slots__
except AttributeError as e:
    print("Error:", e)


## 16. `dataclasses`: Less Boilerplate for Data-Holding Classes

- The `@dataclass` decorator auto-generates `__init__`, `__repr__`, and `__eq__` from type-annotated class attributes — great for simple classes whose main job is holding data.


In [ ]:
from dataclasses import dataclass

@dataclass
class Point3D:
    x: float
    y: float
    z: float = 0.0   # default value


p1 = Point3D(1, 2)
p2 = Point3D(1, 2)

print(p1)              # auto-generated __repr__
print(p1 == p2)          # auto-generated __eq__ (compares field values)


## 17. `isinstance`, `issubclass`, and Duck Typing

- `isinstance(obj, Class)` checks whether an object belongs to a class (or its subclasses).
- `issubclass(SubClass, Class)` checks the class relationship itself.
- **Duck typing**: Python generally cares more about *what an object can do* than *what class it is* — "if it walks like a duck and quacks like a duck, it's a duck." Code that calls `.speak()` on any object doesn't need to check its type first (see the polymorphism example in section 9).


In [ ]:
print(isinstance(d, Duck))
print(isinstance(d, Flyer))
print(issubclass(Duck, Flyer))
print(issubclass(Duck, Animal))  # False, Duck is unrelated to the Animal hierarchy


class Duck2:
    def quack(self):
        return "Quack!"


class Person:
    def quack(self):
        return "I'm pretending to be a duck!"


def make_it_quack(thing):
    # duck typing: no isinstance check needed, we just trust it has .quack()
    return thing.quack()


for entity in [Duck2(), Person()]:
    print(make_it_quack(entity))


## 18. Custom Exceptions

- Exceptions are classes too. Create your own by subclassing `Exception` (directly or indirectly) to represent domain-specific error conditions.


In [ ]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance."""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}, balance is only {balance}")


class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount


acc = BankAccount(100)
try:
    acc.withdraw(150)
except InsufficientFundsError as e:
    print("Error:", e)
    print("Attempted amount:", e.amount)


## 19. Summary: The Four Pillars of OOP

| Pillar | What it means | Where it showed up above |
|---|---|---|
| **Encapsulation** | Bundle data and behavior together; restrict direct access to internal state | `__balance`, `@property` (sections 6-7) |
| **Abstraction** | Expose a simple interface, hide implementation complexity | `ABC`, `@abstractmethod` (section 10) |
| **Inheritance** | Reuse and extend behavior from a parent class | `Dog(Animal)`, `super()` (section 8) |
| **Polymorphism** | The same interface behaves differently depending on the object | `.speak()` across `Dog`/`Cat`/`Cow` (section 9) |
